# Checkerboard Hubbard — figures for the weekly deck

One cell per figure, in slide order. Every path points at `/home/phd25imran/analysis` on **node 251**.

**Data this notebook expects**

| file | contents |
|---|---|
| `data/pairing_master.csv` | pairing **vertex**, L = 8/10/12, U = 0-8, delta = 0-0.7, 6 fillings, 6 seeds |
| `data/magnetic_master.csv` | magnetic observables, same grid |
| `data/pairfull_delta0.csv` | **full and vertex** susceptibility, delta = 0 only |
| `data/sq/sq_L{L}_U{U}.npz` | S(q) grids, 288 records per file |

**A naming trap worth knowing:** in `pairing_master.csv` the columns `chi_son`, `chi_sext`, `chi_d`, `chi_dxy` hold the **vertex**, despite having no `_vertex` suffix. Only `pairfull_delta0.csv` carries both `_full` and `_vertex`.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import glob, os, sys
plt.rcParams['figure.facecolor']='white'

DATA = '/home/phd25imran/analysis/data'
OUTD = '/home/phd25imran/analysis'
for f in ['pairing_master.csv','magnetic_master.csv','pairfull_delta0.csv']:
    p=os.path.join(DATA,f)
    print(('OK   ' if os.path.exists(p) else 'MISSING '), f,
          (str(len(pd.read_csv(p)))+' rows') if os.path.exists(p) else '')
print(('OK   ' if glob.glob(DATA+'/sq/sq_*.npz') else 'MISSING '),
      'sq grids:', len(glob.glob(DATA+'/sq/sq_*.npz')), 'files')

## Slide 5 — Headline: per-site vertex, all four channels, three lattices

The paper's lead result. chi is an unnormalised double site sum and therefore extensive, so it must be divided by N before any comparison across lattice sizes. Once divided, the three lattices collapse onto each other (1% spread at U=4) and only d_xy changes sign.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CSV='/home/phd25imran/analysis/data/pairing_master.csv'
NTARGET, LS, U = 1.0, [8,10,12], 4.0
CH=[('chi_son',r'on-site $s$'),('chi_sext',r'extended $s$'),
    ('chi_d',r'$d_{x^2-y^2}$'),('chi_dxy',r'$d_{xy}$')]
MK={8:'o',10:'s',12:'^'}

d=pd.read_csv(CSV); d=d[d.L.isin(LS)]; d['N']=d.L**2
for c,_ in CH: d[c+'_n']=d[c]/d.N
ns=np.sort(d.n.unique()); N=ns[np.argmin(abs(ns-NTARGET))]
d=d[np.isclose(d.n,N)&np.isclose(d.U,U)]
g=d.groupby(['L','delta'])[[c+'_n' for c,_ in CH]].agg(['mean','sem'])

fig,ax=plt.subplots(1,4,figsize=(18,4.4),facecolor='white')
for a,(c,lab) in zip(ax,CH):
    for L in LS:
        s=g.loc[L]
        a.errorbar(s.index.values,s[(c+'_n','mean')],s[(c+'_n','sem')],
                   marker=MK[L],lw=1.7,ms=6,capsize=2,label=f'$L={L}$')
    a.axhline(0,color='k',lw=0.9,ls=':')
    a.set_title(lab,fontsize=17); a.set_xlabel(r'$\delta$',fontsize=16)
    a.tick_params(labelsize=12)
ax[0].set_ylabel(r'$\chi_\alpha/N$',fontsize=16); ax[0].legend(fontsize=12)
fig.suptitle(rf'Per-site pairing vertex, all four channels, $U={U:g}$, $n={N:.3f}$',fontsize=17)
plt.tight_layout(rect=[0,0,1,0.92]); plt.savefig('/home/phd25imran/analysis/fig_collapse4.png',dpi=105,facecolor='white')

print(f'U={U:g} n={N:.3f}   chi/N   (delta=0 -> peak -> delta=0.7),  collapse spread across L')
for c,lab in CH:
    print(f'\n {lab}')
    for L in LS:
        v=g.loc[L][(c+'_n','mean')].values; dd=g.loc[L].index.values
        print(f'   L={L:2d}: '+'  '.join(f'{x:+.4f}' for x in v))
    v12=g.loc[12][(c+'_n','mean')].values
    sp=[g.loc[L][(c+'_n','mean')].values for L in LS]
    spread=100*np.max([abs(a-b) for a in sp for b in sp])/max(abs(v12).max(),1e-9)
    sign='no sign change' if (v12>0).all() or (v12<0).all() else 'CHANGES SIGN'
    print(f'   delta= '+'  '.join(f'{x:+.4f}'.replace('+','').rjust(7) for x in dd))
    print(f'   -> {sign};  max spread across L = {spread:.0f}% of peak')
plt.show()

## Slides 6 and 7 — Vertex vs U and vs filling, one panel per anisotropy

Two figures from one cell. The first shows d_xy rising through the panels until it is the top curve by delta = 0.4; the second shows the same anisotropy driving it the other way when doped.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CSV='/home/phd25imran/analysis/data/pairing_master.csv'
OUTDIR='/home/phd25imran/analysis'; L=12; UFIX=4.0; NFIX=1.0
CH=[('chi_son',r'on-site $s$','#7f7f7f','o'),('chi_sext',r'extended $s$','#2ca02c','s'),
    ('chi_d',r'$d_{x^2-y^2}$','#1f77b4','^'),('chi_dxy',r'$d_{xy}$','#d62728','D')]
NC=[c+'_n' for c,_,_,_ in CH]

d=pd.read_csv(CSV); d=d[d.L==L]; d['N']=d.L**2
for c,_,_,_ in CH: d[c+'_n']=d[c]/d.N

def panels(fixcol,fixval,xcol,xlabel,fname,suptitle):
    s=d[np.isclose(d[fixcol],fixval)]
    DS=sorted(s.delta.unique())
    sub={dd: s[np.isclose(s.delta,dd)].groupby(xcol)[NC].agg(['mean','sem']) for dd in DS}
    vals=[g[(c,'mean')] for g in sub.values() for c in NC]
    lo=min(v.min() for v in vals); hi=max(v.max() for v in vals); pad=0.06*(hi-lo)
    ncol=4; nrow=int(np.ceil(len(DS)/ncol))
    fig,ax=plt.subplots(nrow,ncol,figsize=(4.7*ncol,4.2*nrow),facecolor='white',squeeze=False)
    for k,dd in enumerate(DS):
        a=ax[k//ncol][k%ncol]; g=sub[dd]
        for c,lab,col,mk in CH:
            a.errorbar(g.index.values,g[(c+'_n','mean')],g[(c+'_n','sem')],
                       marker=mk,ms=6,lw=1.8,color=col,capsize=2.5,label=lab)
        a.axhline(0,color='k',lw=1.3,ls='--')
        a.set_title(rf'$\delta={dd:g}$',fontsize=16)
        a.set_xlabel(xlabel,fontsize=14); a.tick_params(labelsize=11)
        a.set_ylim(lo-pad,hi+pad)
        if k%ncol==0: a.set_ylabel(r'vertex  $\chi_\alpha/N$',fontsize=14)
    for k in range(len(DS),nrow*ncol): ax[k//ncol][k%ncol].axis('off')
    ax[0][0].legend(fontsize=10,loc='best',framealpha=0.95)
    fig.suptitle(suptitle,fontsize=18)
    plt.tight_layout(rect=[0,0,1,0.94],h_pad=2.5)
    plt.savefig(f'{OUTDIR}/{fname}',dpi=100,facecolor='white'); plt.show()
    return sub

s1=panels('n',NFIX,'U','$U$','G1.png',
   rf'Pairing vertex vs interaction, every anisotropy.  $L={L}$, $n={NFIX:g}$')
s2=panels('U',UFIX,'n','Filling $n$','G2.png',
   rf'Pairing vertex vs filling, every anisotropy.  $L={L}$, $U={UFIX:g}$')

print('FIG 1 (n=1): dxy vertex/N at U=8, per delta:')
print('  '+'  '.join(f'd={k:g}:{v.loc[8.0,("chi_dxy_n","mean")]:+.3f}' for k,v in s1.items()))
print('FIG 2 (U=4): dxy vertex/N at n=1.0 and n=0.778, per delta:')
for tag,nn in [('n=1.000',1.0),('n=0.778',0.7777777777777778)]:
    print(f'  {tag}: '+'  '.join(f'd={k:g}:{v.loc[nn,("chi_dxy_n","mean")]:+.3f}' for k,v in s2.items()))

## Slide 8 — Extended s and d_x2-y2 compared directly

Both conventional channels decay with anisotropy, but d_x2-y2 falls about twice as fast. Panel (c) is the difference: extended s only leads near half filling.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
OUT='/home/phd25imran/analysis'
P='/home/phd25imran/analysis/data/'
GRN,BLU='#2ca02c','#1f77b4'

d=pd.read_csv(P+'pairing_master.csv'); d=d[d.L==12]; d['N']=d.L**2
for c in ['chi_sext','chi_d']: d[c+'_n']=d[c]/d.N
f=pd.read_csv(P+'pairfull_delta0.csv'); f=f[f.L==12]
for c in ['sext','d']:
    f['bub_'+c]=f[f'chi_{c}_full']-f[f'chi_{c}_vertex']
    f['rat_'+c]=f[f'chi_{c}_full']/f['bub_'+c]

fig,ax=plt.subplots(1,4,figsize=(19,4.3),facecolor='white')

# (a) vs delta at half filling, all U
s=d[np.isclose(d.n,1.0)&(d.U>0)]
for U,ls in zip([2.,4.,6.,8.],['-',':','--','-.']):
    t=s[np.isclose(s.U,U)].groupby('delta')[['chi_sext_n','chi_d_n']].mean()
    ax[0].plot(t.index,t.chi_sext_n,ls,color=GRN,lw=2,marker='s',ms=5)
    ax[0].plot(t.index,t.chi_d_n,ls,color=BLU,lw=2,marker='^',ms=5)
ax[0].set_xlabel(r'$\delta$',fontsize=15); ax[0].set_ylabel(r'vertex $\chi/N$',fontsize=14)
ax[0].set_title('(a)  both decay with anisotropy'+'\n'+r'(line style: $U=2,4,6,8$)',fontsize=13)
ax[0].plot([],[],color=GRN,lw=2,marker='s',label='extended $s$')
ax[0].plot([],[],color=BLU,lw=2,marker='^',label=r'$d_{x^2-y^2}$')
ax[0].legend(fontsize=11); ax[0].axhline(0,color='k',lw=.9,ls=':')

# (b) vs filling at U=4, several delta
s2=d[np.isclose(d.U,4)]
for dd,al in zip([0.0,0.2,0.4,0.7],[1.0,.75,.5,.3]):
    t=s2[np.isclose(s2.delta,dd)].groupby('n')[['chi_sext_n','chi_d_n']].mean()
    ax[1].plot(t.index,t.chi_sext_n,color=GRN,alpha=al,lw=2,marker='s',ms=5)
    ax[1].plot(t.index,t.chi_d_n,color=BLU,alpha=al,lw=2,marker='^',ms=5)
ax[1].axhline(0,color='k',lw=1.1,ls='--')
ax[1].set_xlabel('Filling $n$',fontsize=15)
ax[1].set_title(r'(b)  both turn attractive near $n=1$'+'\n'+r'(fading: $\delta=0,0.2,0.4,0.7$)',fontsize=13)

# (c) which leads: difference
for dd,al in zip([0.0,0.2,0.4,0.7],[1.0,.75,.5,.3]):
    t=s2[np.isclose(s2.delta,dd)].groupby('n')[['chi_sext_n','chi_d_n']].mean()
    ax[2].plot(t.index,t.chi_sext_n-t.chi_d_n,color='#6D2E46',alpha=al,lw=2,marker='o',ms=5,
               label=rf'$\delta={dd:g}$')
ax[2].axhline(0,color='k',lw=1.1,ls='--')
ax[2].set_xlabel('Filling $n$',fontsize=15)
ax[2].set_ylabel(r'$\chi_{s\text{-}ext}-\chi_{d_{x^2-y^2}}$',fontsize=13)
ax[2].set_title(r'(c)  extended $s$ leads only near $n=1$',fontsize=14); ax[2].legend(fontsize=10,loc='lower right')

# (d) P/Pbar at delta=0
g=f[np.isclose(f.n,1.0)].groupby('U')[['rat_sext','rat_d']].agg(['mean','sem'])
ax[3].errorbar(g.index,g[('rat_sext','mean')],g[('rat_sext','sem')],color=GRN,lw=2,
               marker='s',ms=7,capsize=3,label='extended $s$')
ax[3].errorbar(g.index,g[('rat_d','mean')],g[('rat_d','sem')],color=BLU,lw=2,
               marker='^',ms=7,capsize=3,label=r'$d_{x^2-y^2}$')
ax[3].axhline(1,color='k',lw=1.1,ls='--')
ax[3].set_xlabel('$U$',fontsize=15); ax[3].set_ylabel(r'$P/\bar{P}$',fontsize=14)
ax[3].set_title(r'(d)  $\delta=0$: they cross near $U=6$',fontsize=14); ax[3].legend(fontsize=11)
for a in ax: a.tick_params(labelsize=11)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_sd.png',dpi=110,facecolor='white')

print('crossover: extended s minus dx2-y2 at half filling, per delta (U=4)')
for dd in sorted(s2.delta.unique()):
    t=s2[np.isclose(s2.delta,dd)].groupby('n')[['chi_sext_n','chi_d_n']].mean()
    print(f'  d={dd:g}: sext={t.chi_sext_n.loc[1.0]:+.4f}  d={t.chi_d_n.loc[1.0]:+.4f}  diff={t.chi_sext_n.loc[1.0]-t.chi_d_n.loc[1.0]:+.4f}')
print()
print('P/Pbar at half filling, delta=0:')
for U in g.index: print(f'  U={U:g}: sext={g.loc[U,("rat_sext","mean")]:.3f}  d={g.loc[U,("rat_d","mean")]:.3f}')
plt.show()

## Slide 11 — (n, delta) phase diagram, every channel and interaction

One shared colour scale across all twenty panels, as requested. It has to be symmetric-log: on-site s reaches -0.59 while the d channels live at +/-0.04, so a linear shared scale flattens them to white. Set norm=None and vmin/vmax to see that for yourself.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
OUT='/home/phd25imran/analysis'
CSV='/home/phd25imran/analysis/data/pairing_master.csv'
L=12
CH=[('chi_son',r'on-site $s$'),('chi_sext',r'extended $s$'),
    ('chi_d',r'$d_{x^2-y^2}$'),('chi_dxy',r'$d_{xy}$')]

d=pd.read_csv(CSV); d=d[d.L==L]; d['N']=d.L**2
for c,_ in CH: d[c+'_n']=d[c]/d.N
US=sorted(d.U.unique())
g=d.groupby(['U','n','delta'])[[c+'_n' for c,_ in CH]].mean().reset_index()

# ONE shared symmetric scale across every panel. 2nd/98th percentile over all
# channels and all U, so the on-site-s extreme does not flatten everything else.
allv=np.concatenate([g[c+'_n'].values for c,_ in CH])
lim=float(np.max(np.abs(allv)))
# ONE shared scale, but symmetric-log: the on-site-s extreme reaches -0.59 while the
# physics in the d channels lives at +/-0.04. A linear shared scale flattens the latter
# to white. linthresh sets where the scale turns from linear to logarithmic.
NORM=SymLogNorm(linthresh=0.01, vmin=-lim, vmax=lim, base=10)
print(f'shared range +/-{lim:.4f}, linear below 0.01  (raw {allv.min():+.3f} to {allv.max():+.3f})')

ns=np.sort(g.n.unique()); ds=np.sort(g.delta.unique())
def edges(v):
    v=np.asarray(v,float); m=(v[1:]+v[:-1])/2
    return np.r_[v[0]-(m[0]-v[0]), m, v[-1]+(v[-1]-m[-1])]
NE,DE=edges(ns),edges(ds)

fig,ax=plt.subplots(len(US),4,figsize=(15.5,3.15*len(US)),facecolor='white',squeeze=False)
for i,U in enumerate(US):
    for j,(c,lab) in enumerate(CH):
        a=ax[i][j]
        piv=g[np.isclose(g.U,U)].pivot(index='delta',columns='n',values=c+'_n')
        im=a.pcolormesh(NE,DE,piv.values,cmap='RdBu_r',norm=NORM,shading='flat')
        a.set_xticks([0.5,0.7,0.9]); a.set_yticks([0,0.2,0.4,0.6])
        a.tick_params(labelsize=10)
        if i==0: a.set_title(lab,fontsize=16)
        if j==0: a.set_ylabel(f'$U={U:g}$\n'+r'$\delta$',fontsize=13)
        else: a.set_yticklabels([])
        if i==len(US)-1: a.set_xlabel('Filling $n$',fontsize=13)
        else: a.set_xticklabels([])
cb=fig.colorbar(im,ax=ax,fraction=0.016,pad=0.015)
cb.set_label(r'pairing vertex  $\chi_\alpha/N$   (symlog)   red = attractive,  blue = repulsive',fontsize=12)
fig.suptitle(rf'$(n,\delta)$ phase diagram, all channels and interactions, $L={L}$  —  one shared colour scale',
             fontsize=17)
plt.savefig(f'{OUT}/fig_pd.png',dpi=105,facecolor='white',bbox_inches='tight')
print('nothing clipped; full range shown')
plt.show()

## Slide 12 — Full P against the uncorrelated P_bub (White comparison)

Only available at delta = 0, which is the limit where the square-lattice benchmarks apply. The gap between the curves is what the interaction does; the height of the curves is mostly kinematics.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CSV='/home/phd25imran/analysis/data/pairfull_delta0.csv'
L, UFIX, NFIX = 12, 4.0, 1.0
CH=[('son',r'on-site $s$'),('sext',r'extended $s$'),
    ('d',r'$d_{x^2-y^2}$'),('dxy',r'$d_{xy}$')]

d=pd.read_csv(CSV); d=d[d.L==L]
for c,_ in CH:
    d[f'bub_{c}']=d[f'chi_{c}_full']-d[f'chi_{c}_vertex']

fig,ax=plt.subplots(2,4,figsize=(18,8.4),facecolor='white')

# row 1: vs U at half filling
s=d[np.isclose(d.n,NFIX)]
g=s.groupby('U')[[f'chi_{c}_full' for c,_ in CH]+[f'bub_{c}' for c,_ in CH]
                 +[f'chi_{c}_vertex' for c,_ in CH]].agg(['mean','sem'])
for j,(c,lab) in enumerate(CH):
    a=ax[0,j]; U=g.index.values
    a.errorbar(U,g[(f'chi_{c}_full','mean')],g[(f'chi_{c}_full','sem')],
               marker='o',ms=8,lw=2,color='C3',capsize=3,label=r'full  $P$')
    a.plot(U,g[(f'bub_{c}','mean')],marker='o',ms=8,lw=2,color='C0',
           mfc='white',mew=2,ls='--',label=r'bubble  $\bar P$')
    a.set_title(lab,fontsize=17); a.set_xlabel('$U$',fontsize=15)
    a.tick_params(labelsize=12)
    v=g[(f'chi_{c}_vertex','mean')].values
    a.text(0.05,0.05,'vertex '+('>0 attractive' if v[1:].mean()>0 else '<0 repulsive'),
           transform=a.transAxes,fontsize=12,
           color='darkred' if v[1:].mean()>0 else 'navy')
ax[0,0].set_ylabel(rf'$\chi$   ($n={NFIX:g}$)',fontsize=15); ax[0,0].legend(fontsize=12)

# row 2: vs filling at fixed U
s2=d[np.isclose(d.U,UFIX)]
g2=s2.groupby('n')[[f'chi_{c}_full' for c,_ in CH]+[f'bub_{c}' for c,_ in CH]].agg(['mean','sem'])
for j,(c,lab) in enumerate(CH):
    a=ax[1,j]; nn=g2.index.values
    a.errorbar(nn,g2[(f'chi_{c}_full','mean')],g2[(f'chi_{c}_full','sem')],
               marker='s',ms=7,lw=2,color='C3',capsize=3,label=r'full  $P$')
    a.plot(nn,g2[(f'bub_{c}','mean')],marker='s',ms=7,lw=2,color='C0',
           mfc='white',mew=2,ls='--',label=r'bubble  $\bar P$')
    a.set_xlabel('Filling $n$',fontsize=15); a.tick_params(labelsize=12)
ax[1,0].set_ylabel(rf'$\chi$   ($U={UFIX:g}$)',fontsize=15); ax[1,0].legend(fontsize=12)
fig.suptitle(rf'Full vs uncorrelated pair-field susceptibility, $L={L}$, $\delta=0$'
             '\n'+r'$P>\bar P$ means the interaction is attractive in that channel',fontsize=17)
plt.tight_layout(rect=[0,0,1,0.91]); plt.savefig('/home/phd25imran/analysis/fig_fullsus.png',dpi=105,facecolor='white')

print(f'L={L}, delta=0, n={NFIX:g}:   full / bubble / vertex')
for c, lab in CH:
    for u in g.index:
        if u == 0: continue
        f = g.loc[u, (f"chi_{c}_full", "mean")]
        b = g.loc[u, (f"bub_{c}", "mean")]
        v = g.loc[u, (f"chi_{c}_vertex", "mean")]
        print(f'  {c:5s} U={u:g}:  full {f:7.2f}   bubble {b:7.2f}   vertex {v:+7.2f}')
plt.show()

## Slide 13 — P / P_bub: all four channels on one axis

Dividing by the bubble removes the kinematic factor that made the y-axes differ by 6x. Above 1 the interaction is attractive. All four curves start at exactly 1.000 at U = 0.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CSV='/home/phd25imran/analysis/data/pairfull_delta0.csv'
L, UFIX, NFIX = 12, 4.0, 1.0
CH=[('son',r'on-site $s$','#7f7f7f','o'),('sext',r'extended $s$','#2ca02c','s'),
    ('d',r'$d_{x^2-y^2}$','#1f77b4','^'),('dxy',r'$d_{xy}$','#d62728','D')]

d=pd.read_csv(CSV); d=d[d.L==L]
for c,_,_,_ in CH:
    d[f'bub_{c}']=d[f'chi_{c}_full']-d[f'chi_{c}_vertex']
    d[f'rat_{c}']=d[f'chi_{c}_full']/d[f'bub_{c}']

fig,ax=plt.subplots(1,2,figsize=(14,5.6),facecolor='white')

s=d[np.isclose(d.n,NFIX)]
g=s.groupby('U')[[f'rat_{c}' for c,_,_,_ in CH]].agg(['mean','sem'])
for c,lab,col,mk in CH:
    a=g.index.values
    ax[0].errorbar(a,g[(f'rat_{c}','mean')],g[(f'rat_{c}','sem')],
                   marker=mk,ms=8,lw=2,color=col,capsize=3,label=lab)
ax[0].set_xlabel('$U$',fontsize=17); ax[0].set_title(rf'at half filling $n={NFIX:g}$',fontsize=16)

s2=d[np.isclose(d.U,UFIX)]
g2=s2.groupby('n')[[f'rat_{c}' for c,_,_,_ in CH]].agg(['mean','sem'])
for c,lab,col,mk in CH:
    nn=g2.index.values
    ax[1].errorbar(nn,g2[(f'rat_{c}','mean')],g2[(f'rat_{c}','sem')],
                   marker=mk,ms=8,lw=2,color=col,capsize=3,label=lab)
ax[1].set_xlabel('Filling $n$',fontsize=17); ax[1].set_title(rf'at $U={UFIX:g}$',fontsize=16)

for a in ax:
    a.axhline(1,color='k',lw=1.4,ls='--')
    a.tick_params(labelsize=13); a.set_ylabel(r'$P/\bar{P}$',fontsize=17)
    a.legend(fontsize=12, loc='center right', framealpha=0.95)
    a.text(0.02, 0.97, 'above 1: interaction ATTRACTIVE', transform=a.transAxes,
           fontsize=11, va='top', color='darkred')
    a.text(0.02, 0.03, 'below 1: repulsive', transform=a.transAxes,
           fontsize=11, va='bottom', color='navy')
fig.suptitle(rf'All four pairing channels on one scale, $L={L}$, $\delta=0$',fontsize=18)
plt.tight_layout(rect=[0,0,1,0.93]); plt.savefig('/home/phd25imran/analysis/fig_ratio.png',dpi=105,facecolor='white')

print('P/Pbar at half filling:   (>1 attractive, <1 repulsive)')
hdr = ''.join(f'{c:>14}' for c, _, _, _ in CH)
print(f'{"U":>4}{hdr}')
for u in g.index:
    print(f'{u:4g}' + ''.join(f'{g.loc[u, (f"rat_{c}", "mean")]:14.3f}' for c, _, _, _ in CH))
print()
print('P/Pbar vs filling at U=4:')
print(f'{"n":>6}{hdr}')
for n in g2.index:
    print(f'{n:6.3f}' + ''.join(f'{g2.loc[n, (f"rat_{c}", "mean")]:14.3f}' for c, _, _, _ in CH))
plt.show()

## Slide 14 — Magnetic order parameter against U and delta (meeting point 7)

Delta m = m - m(U=0) isolates the interaction-induced moment, since S(q) is finite even for free fermions. U builds the moment, delta suppresses it, and the product plateaus.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CSV  = '/home/phd25imran/analysis/data/magnetic_master.csv'
L, NTARGET = 12, 1.0

d = pd.read_csv(CSV); d = d[d.L == L]
g = d.groupby(['U','n','delta']).m.agg(['mean','sem']).reset_index().rename(columns={'mean':'m','sem':'e'})
u0 = g[g.U == 0][['n','delta','m','e']].rename(columns={'m':'m0','e':'e0'})
g = g.merge(u0, on=['n','delta'])
g['dm']  = g.m - g.m0
g['err'] = np.hypot(g.e, g.e0)
g['mAM'] = g.dm * g.delta
g['mAM_err'] = g.err * g.delta

ns = np.sort(g.n.unique()); N = ns[np.argmin(abs(ns - NTARGET))]
s  = g[np.isclose(g.n, N)]
US = sorted(u for u in s.U.unique() if u > 0); DS = np.sort(s.delta.unique())

fig, ax = plt.subplots(2, 2, figsize=(13, 10), facecolor='white')

P = s.pivot(index='delta', columns='U', values='mAM')[US]
im = ax[0,0].pcolormesh(np.arange(len(US)+1), np.r_[DS - 0.05, DS[-1]+0.05], P.values,
                        cmap='magma', shading='flat')
ax[0,0].set_xticks(np.arange(len(US))+0.5); ax[0,0].set_xticklabels([f'{u:g}' for u in US])
ax[0,0].set_yticks(DS)
ax[0,0].set_xlabel('$U$', fontsize=16); ax[0,0].set_ylabel(r'$\delta$', fontsize=17)
ax[0,0].set_title(r'(a)  $m_{\rm AM}=\Delta m\,\delta$', fontsize=16)
fig.colorbar(im, ax=ax[0,0])

for dd, c in zip(DS, plt.cm.viridis(np.linspace(0,.9,len(DS)))):
    t = s[np.isclose(s.delta, dd)].sort_values('U')
    ax[0,1].errorbar(t.U, t.dm, t.err, marker='o', color=c, lw=1.8, ms=5,
                     capsize=2, label=rf'$\delta={dd:g}$')
ax[0,1].set_xlabel('$U$', fontsize=16); ax[0,1].set_ylabel(r'$\Delta m$', fontsize=16)
ax[0,1].set_title(r'(b)  interaction builds the moment', fontsize=16)
ax[0,1].legend(fontsize=9, ncol=2)

for u, c in zip(US, plt.cm.plasma(np.linspace(0,.8,len(US)))):
    t = s[np.isclose(s.U, u)].sort_values('delta')
    ax[1,0].errorbar(t.delta, t.dm, t.err, marker='s', color=c, lw=1.8, ms=5,
                     capsize=2, label=f'$U={u:g}$')
ax[1,0].set_xlabel(r'$\delta$', fontsize=17); ax[1,0].set_ylabel(r'$\Delta m$', fontsize=16)
ax[1,0].set_title(r'(c)  anisotropy suppresses it', fontsize=16)
ax[1,0].legend(fontsize=11)

for u, c in zip(US, plt.cm.plasma(np.linspace(0,.8,len(US)))):
    t = s[np.isclose(s.U, u)].sort_values('delta')
    ax[1,1].errorbar(t.delta, t.mAM, t.mAM_err, marker='^', color=c, lw=1.8, ms=6,
                     capsize=2, label=f'$U={u:g}$')
ax[1,1].set_xlabel(r'$\delta$', fontsize=17); ax[1,1].set_ylabel(r'$m_{\rm AM}$', fontsize=16)
ax[1,1].set_title(r'(d)  the product', fontsize=16)
ax[1,1].legend(fontsize=11)
for a in ax.ravel(): a.tick_params(labelsize=12)
fig.suptitle(rf'Altermagnetic order parameter, $L={L}$, $n={N:.3f}$', fontsize=18)
plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig('/home/phd25imran/analysis/fig_mam.png', dpi=105, facecolor='white')
print('n =', N)
print(s[s.U>0].groupby('U').apply(lambda t: f"dm({DS[0]:g})={t[np.isclose(t.delta,DS[0])].dm.values[0]:.4f} -> dm(0.7)={t[np.isclose(t.delta,0.7)].dm.values[0]:.4f}", include_groups=False).to_string())
plt.show()

## Slide 15a — Ordering wavevector correction

Recomputes the moment at the measured peak of S(q) rather than at a fixed (pi,pi). The suppression survives: factor 2.3 instead of 3.0.

In [ ]:
import numpy as np, glob, os, pandas as pd, matplotlib.pyplot as plt

SQDIR='/home/phd25imran/analysis/data/sq'
L, NTARGET = 12, 1.0

rows=[]
for f in sorted(glob.glob(os.path.join(SQDIR,'sq_*.npz'))):
    z=np.load(f,allow_pickle=True); cols=[str(c) for c in z['cols']]
    ix={c:i for i,c in enumerate(cols)}; meta=z['meta']; Sq=z['Sq']
    for k in range(len(meta)):
        r=meta[k]; LL=int(r[ix['L']]); S=Sq[k]; h=LL//2
        i,j=np.unravel_index(np.argmax(S),S.shape)
        rows.append(dict(L=LL,U=float(r[ix['U']]),delta=float(r[ix['delta']]),
                         n=float(r[ix['n']]),seed=int(r[ix['seed']]),
                         Spipi=float(S[h,h]), Sstar=float(S[i,j]),
                         qx=2*i/LL, qy=2*j/LL))
D=pd.DataFrame(rows)
D['m_pipi']=np.sqrt(np.maximum(D.Spipi,0)/D.L**2)
D['m_star']=np.sqrt(np.maximum(D.Sstar,0)/D.L**2)

def build(col):
    g=D.groupby(['L','U','n','delta'])[col].agg(['mean','sem']).reset_index() \
       .rename(columns={'mean':'m','sem':'e'})
    u0=g[g.U==0][['L','n','delta','m','e']].rename(columns={'m':'m0','e':'e0'})
    g=g.merge(u0,on=['L','n','delta']); g['dm']=g.m-g.m0; g['err']=np.hypot(g.e,g.e0)
    return g
Gp, Gs = build('m_pipi'), build('m_star')

ns=np.sort(D.n.unique()); N=ns[np.argmin(abs(ns-NTARGET))]
sp=Gp[(Gp.L==L)&np.isclose(Gp.n,N)]; ss=Gs[(Gs.L==L)&np.isclose(Gs.n,N)]
US=[2.,4.,6.,8.]; DS=np.sort(sp.delta.unique())

fig,ax=plt.subplots(1,3,figsize=(16.5,4.8),facecolor='white')
for u,c in zip(US, plt.cm.plasma(np.linspace(0,.8,len(US)))):
    t=sp[np.isclose(sp.U,u)].sort_values('delta')
    ax[0].errorbar(t.delta,t.dm,t.err,marker='s',color=c,lw=1.7,ms=5,capsize=2,label=f'$U={u:g}$')
    t=ss[np.isclose(ss.U,u)].sort_values('delta')
    ax[1].errorbar(t.delta,t.dm,t.err,marker='o',color=c,lw=1.7,ms=5,capsize=2,label=f'$U={u:g}$')
ax[0].set_title(r'(a)  $\Delta m$ at fixed $(\pi,\pi)$',fontsize=15)
ax[1].set_title(r'(b)  $\Delta m$ at the true peak $q^*$',fontsize=15)
for a in ax[:2]:
    a.set_xlabel(r'$\delta$',fontsize=16); a.set_ylabel(r'$\Delta m$',fontsize=15)
    a.legend(fontsize=10); a.tick_params(labelsize=11)

sub=D[(D.L==L)&np.isclose(D.n,N)&(D.U>0)]
q=sub.groupby('delta').apply(lambda t:(t.Spipi/t.Sstar).mean(),include_groups=False)
ax[2].plot(q.index,q.values,marker='D',color='crimson',lw=2,ms=7)
ax[2].axhline(1,color='k',lw=0.9,ls=':')
ax[2].axhline(0.95,color='grey',lw=0.9,ls='--')
ax[2].set_xlabel(r'$\delta$',fontsize=16)
ax[2].set_ylabel(r'$S(\pi,\pi)\,/\,S(q^*)$',fontsize=15)
ax[2].set_title(r'(c)  is $(\pi,\pi)$ still the peak?',fontsize=15)
ax[2].tick_params(labelsize=11)
fig.suptitle(rf'Ordering wavevector, $L={L}$, $n={N:.3f}$',fontsize=17)
plt.tight_layout(rect=[0,0,1,0.93]); plt.savefig('/home/phd25imran/analysis/fig_qstar.png',dpi=105,facecolor='white')

print(f'L={L} n={N:.3f}   fraction of magnetic weight at (pi,pi), and modal q*')
for d in DS:
    t=sub[np.isclose(sub.delta,d)]
    mode=t.groupby(['qx','qy']).size().idxmax()
    print(f'  delta={d:g}:  S(pi,pi)/S(q*) = {(t.Spipi/t.Sstar).mean():.3f}   modal q* = ({mode[0]:.2f}pi,{mode[1]:.2f}pi)')
print()
for u in US:
    a=sp[np.isclose(sp.U,u)].sort_values('delta').dm.values
    b=ss[np.isclose(ss.U,u)].sort_values('delta').dm.values
    print(f'U={u:g}: dm(pi,pi) {a[0]:.4f}->{a[-1]:.4f} (x{a[0]/a[-1]:.1f})   dm(q*) {b[0]:.4f}->{b[-1]:.4f} (x{b[0]/b[-1]:.1f})')
plt.show()

## Slide 15b — Spin structure factor maps

Needs the merged S(q) grids in ~/analysis/data/sq (one file per L and U, 288 records each). The cyan cross marks the true peak, the white plus marks (pi,pi).

In [ ]:
import numpy as np, glob, os, matplotlib.pyplot as plt

OUT='/home/phd25imran/analysis'
SQDIRS=['/home/phd25imran/analysis/data/sq']
L, U, NTARGET = 12, 4.0, 1.0

recs=[]
for Dd in SQDIRS:
    for f in sorted(glob.glob(os.path.join(Dd,'sq_*.npz'))):
        z=np.load(f,allow_pickle=True)
        cols=[str(c) for c in z['cols']]; meta=z['meta']; Sq=z['Sq']
        ix={c:i for i,c in enumerate(cols)}
        for k in range(len(meta)):
            recs.append(dict(L=int(meta[k][ix['L']]),U=float(meta[k][ix['U']]),
                             delta=float(meta[k][ix['delta']]),
                             n=float(meta[k][ix['n']]),Sq=Sq[k]))
inv={}
for r in recs: inv.setdefault((r['L'],r['U']),set()).add(r['delta'])
print('AVAILABLE (L,U) -> deltas:')
for k in sorted(inv): print(f'   L={k[0]:2d} U={k[1]:g}: {sorted(inv[k])}')

sel=[r for r in recs if r['L']==L and np.isclose(r['U'],U)]
if not sel: raise SystemExit(f'no grids for L={L} U={U:g}')
ns=sorted({r['n'] for r in sel}); N=min(ns,key=lambda x:abs(x-NTARGET))
sel=[r for r in sel if np.isclose(r['n'],N)]
DS=sorted({r['delta'] for r in sel})
avg={d:np.mean([r['Sq'] for r in sel if np.isclose(r['delta'],d)],axis=0) for d in DS}

vmax=max(v.max() for v in avg.values())
step=2.0/L                                   # cell width in units of pi
ext=[-step/2, 2-step/2, -step/2, 2-step/2]   # q/pi from 0 to 2, cell-centred

fig,ax=plt.subplots(1,len(DS),figsize=(2.45*len(DS),3.3),facecolor='white',squeeze=False)
for a,d in zip(ax[0],DS):
    S=avg[d]
    im=a.imshow(S.T,origin='lower',extent=ext,cmap='inferno',vmin=0,vmax=vmax,
                interpolation='nearest',aspect='equal')
    i,j=np.unravel_index(np.argmax(S),S.shape)
    a.plot(2*i/L,2*j/L,marker='x',color='cyan',ms=10,mew=2.5)
    a.plot(1,1,marker='+',color='w',ms=9,mew=1.4)      # (pi,pi) for reference
    a.set_title(rf'$\delta={d:g}$',fontsize=14)
    a.set_xticks([0,1,2]); a.set_yticks([0,1,2])
    a.set_xticklabels(['0',r'$\pi$',r'$2\pi$'],fontsize=10)
    a.set_yticklabels(['0',r'$\pi$',r'$2\pi$'] if a is ax[0,0] else [],fontsize=10)
    a.set_xlabel(r'$q_x$',fontsize=12)
ax[0,0].set_ylabel(r'$q_y$',fontsize=12)
fig.colorbar(im,ax=ax[0].tolist(),fraction=0.015,pad=0.015,label=r'$S^{zz}(q)$')
fig.suptitle(rf'Spin structure factor, $L={L}$, $U={U:g}$, $n={N:.3f}$   '
             r'($\times$ peak,  $+$ is $(\pi,\pi)$)',fontsize=14)
plt.savefig('/home/phd25imran/analysis/fig_sqmap3.png',dpi=110,facecolor='white',bbox_inches='tight')

h=L//2
print('\ndelta   peak q             S_max    S(pi,pi)  ratio')
for d in DS:
    S=avg[d]; i,j=np.unravel_index(np.argmax(S),S.shape)
    print(f' {d:4.1f}   ({2*i/L:.2f}pi,{2*j/L:.2f}pi)    {S[i,j]:.4f}   {S[h,h]:.4f}   {S[h,h]/S[i,j]:.3f}')
plt.show()

## Bonus — 1:2:1 twist average (run once the (-1,-1) leg lands)

Takes two positional arguments: the (-1,+1) csv and the (-1,-1) csv. In the notebook, set sys.argv before running, or edit T21/T11 directly.

In [ ]:
"""1:2:1 twist average of the pairing vertex. Run once the (-1,-1) leg lands."""
import numpy as np, pandas as pd, glob, os

PER = '/home/phd25imran/analysis/data/pairing_master.csv'   # periodic, weight 1
# Point these at the two twist legs once they are collected onto 251.
# The periodic leg (weight 1) is already inside pairing_master.csv.
T21 = '/home/phd25imran/analysis/data/twist/twist_L12_U4_apx-1apy1.csv'    # (-1,+1), weight 2
T11 = '/home/phd25imran/analysis/data/twist/twist_L12_U4_apx-1apy-1.csv'   # (-1,-1), weight 1
L, U, NUP = 12, 4.0, 72
CH = ['chi_son', 'chi_sext', 'chi_d', 'chi_dxy']
N = L * L

missing = [f for f in (T21, T11) if not os.path.exists(f)]
if missing:
    print('Twist legs not collected yet. Missing:')
    for f in missing: print('   ', f)
    print('\nThe (-1,+1) leg is on node 250 in ~/twist/, the (-1,-1) leg on 254 in')
    print('~/twist_fast/d*/. Copy them to ~/analysis/data/twist/ and re-run this cell.')
    raise SystemExit

p = pd.read_csv(PER)
p = p[(p.L == L) & np.isclose(p.U, U) & (p.nup == NUP)]
per = p.groupby('delta')[CH].agg(['mean', 'sem']) / N       # these columns ARE the vertex

def leg(path):
    d = pd.concat([pd.read_csv(f) for f in glob.glob(path)], ignore_index=True) \
        if '*' in path else pd.read_csv(path)
    d = d[(d.L == L) & np.isclose(d.U, U) & (d.nup == NUP)]
    d = d.rename(columns={c + '_vertex': c for c in CH})
    return d.groupby('delta')[CH].agg(['mean', 'sem']) / N

a, b = leg(T21), leg(T11)
ds = sorted(set(per.index) & set(a.index) & set(b.index))
print(f'twist-averaged pairing vertex  chi/N   L={L}, U={U:g}, n=1.0')
print(f'weights 1 : 2 : 1  for  (+1,+1) : (-1,+1) : (-1,-1)\n')
print(f'{"delta":>6}{"periodic":>11}{"(-1,+1)":>11}{"(-1,-1)":>11}{"AVERAGE":>12}{"+/-":>9}   swap?')
for x in ds:
    v = [per.loc[x, ('chi_dxy', 'mean')], a.loc[x, ('chi_dxy', 'mean')], b.loc[x, ('chi_dxy', 'mean')]]
    e = [per.loc[x, ('chi_dxy', 'sem')], a.loc[x, ('chi_dxy', 'sem')], b.loc[x, ('chi_dxy', 'sem')]]
    avg = (v[0] + 2 * v[1] + v[2]) / 4
    err = np.sqrt(e[0]**2 + (2 * e[1])**2 + e[2]**2) / 4
    print(f'{x:6.1f}{v[0]:+11.4f}{v[1]:+11.4f}{v[2]:+11.4f}{avg:+12.4f}{err:9.4f}'
          f'   {"ATTRACTIVE" if avg > 2*err else ("repulsive" if avg < -2*err else "unresolved")}')

print('\nall four channels, twist-averaged:')
print(f'{"delta":>6}' + ''.join(f'{c.replace("chi_",""):>12}' for c in CH))
for x in ds:
    row = ''
    for c in CH:
        avg = (per.loc[x, (c, 'mean')] + 2 * a.loc[x, (c, 'mean')] + b.loc[x, (c, 'mean')]) / 4
        row += f'{avg:+12.4f}'
    print(f'{x:6.1f}{row}')
plt.show()